# siope_uscite_comuni - notebook v0

Validazione della pipeline per fasi: **raw → clean → mart**.

- scopo: verificare che ogni layer sia corretto e coerente con il precedente
- le SQL sono la fonte di verità: i check numerici devono essere letti alla luce di quello che dichiarano
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit

In [ ]:
import reimport yamlimport pandas as pdfrom pathlib import Path# --- Unici parametri da impostare manualmente ---METRICA       = "importo_totale"  # colonna numerica nel mart (tabella agg)
METRICA_CLEAN = "importo"        # colonna nel clean
# --- Anchor: usa il path del notebook se disponibile (VSCode), altrimenti cwd ---try:    _start = Path(__vsc_ipynb_file__).resolve().parent  # VSCode Jupyterexcept NameError:    _start = Path.cwd().resolve()# Walk up finché non troviamo dataset.ymlcandidate_dir = Nonefor probe in [_start, *_start.parents]:    if (probe / "dataset.yml").exists():        candidate_dir = probe        breakif candidate_dir is None:    raise FileNotFoundError(f"dataset.yml non trovato risalendo da {_start}")cfg = yaml.safe_load((candidate_dir / "dataset.yml").read_text())SLUG       = cfg["dataset"]["name"]ANNO_RUN   = cfg["dataset"]["years"][-1]MART_TABLE = "siope_uscite_comuni_agg_labeled"  # aggiornato: _agg non piu` generatoENCODING   = cfg.get("clean", {}).get("read", {}).get("encoding", "utf-8")DELIM      = cfg.get("clean", {}).get("read", {}).get("delim", ",")HEADER     = cfg.get("clean", {}).get("read", {}).get("header", True)SKIP       = cfg.get("clean", {}).get("read", {}).get("skip", 0)DI_ROOT   = (candidate_dir / cfg["root"]).resolve()RAW_DIR   = DI_ROOT / "data" / "raw"   / SLUG / str(ANNO_RUN)CLEAN_DIR = DI_ROOT / "data" / "clean" / SLUG / str(ANNO_RUN)MART_DIR  = DI_ROOT / "data" / "mart"  / SLUG / str(ANNO_RUN)SQL_DIR   = candidate_dir / "sql"print(f"slug      : {SLUG}")print(f"anno_run  : {ANNO_RUN}")print(f"mart table: {MART_TABLE}")print(f"encoding  : {ENCODING}  delim: {repr(DELIM)}")print(f"header    : {HEADER}  skip: {SKIP}")

## SQL di riferimento

Le SQL sono la fonte di verità per capire cosa deve contenere ogni layer.
Leggile prima di interpretare i check numerici.

In [ ]:
for sql_file in sorted(SQL_DIR.glob("*.sql")):
    print(f"{'='*60}")
    print(f"  {sql_file.name}")
    print(f"{'='*60}")
    print(sql_file.read_text())
    print()

## 1. Raw

Verifica che il file raw esista, sia leggibile con i parametri dichiarati in `dataset.yml` e abbia il numero di righe atteso.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv")) + sorted(RAW_DIR.glob("*.xlsx")) + sorted(RAW_DIR.glob("*.parquet"))
if not raw_files:
    raise FileNotFoundError(f"Nessun file raw trovato in {RAW_DIR}")

raw_path = raw_files[0]
print(f"File: {raw_path.name}  ({raw_path.stat().st_size / 1024:.0f} KB)")

N_RAW = None
raw_full = None
try:
    if raw_path.suffix == ".parquet":
        raw_full = pd.read_parquet(raw_path)
    elif raw_path.suffix in (".csv", ".tsv"):
        header_row = 0 if HEADER else None
        raw_full = pd.read_csv(raw_path, encoding=ENCODING, sep=DELIM, header=header_row, skiprows=SKIP)
    elif raw_path.suffix == ".xlsx":
        raw_full = pd.read_excel(raw_path, header=0 if HEADER else None, skiprows=SKIP)
    N_RAW = len(raw_full)
    print(f"Righe raw   : {N_RAW}")
    print(f"Colonne raw : {len(raw_full.columns)} -> {list(raw_full.columns)}")
    print("Raw caricato OK")
    raw_full.head(3)
except Exception as e:
    print(f"WARNING: raw non leggibile con pandas ({e})")
    print("-> Skip raw-load, il clean parquet e gia validato dalla pipeline.")
    N_RAW = None

## 2. Clean

Verifica schema e null. I null su colonne `try_cast` sono attesi se il raw contiene valori non parsabili.
Il confronto righe raw vs clean mostra quante righe sono state filtrate dal `WHERE` della clean.sql.

In [ ]:
clean_files = sorted(CLEAN_DIR.glob("*.parquet"))
if not clean_files:
    raise FileNotFoundError(f"Clean non trovato in {CLEAN_DIR}. Eseguire: toolkit run clean")

clean = pd.read_parquet(clean_files[0])
N_CLEAN = len(clean)

print(f"Righe clean : {N_CLEAN}")
print(f"Colonne     : {list(clean.columns)}")
clean.head(3)

In [ ]:
if N_RAW is not None:
    dropped = N_RAW - N_CLEAN
    dropped_pct = dropped / N_RAW * 100 if N_RAW > 0 else 0
    print(f"Righe raw         : {N_RAW}")
    print(f"Righe clean       : {N_CLEAN}")
    print(f"Righe filtrate    : {dropped} ({dropped_pct:.1f}%)")
    print()
    if dropped > 0:
        print("-> Verificare la WHERE in clean.sql per capire quali righe vengono escluse.")
    else:
        print("-> Nessuna riga filtrata: clean e fedele al raw.")
else:
    print(f"Raw non caricato (parsing error) -- righe clean: {N_CLEAN}")
    print("-> Check raw-vs-clean saltato. Clean gia validato dalla pipeline.")

print("\nNull per colonna clean:")
null_counts = clean.isnull().sum()
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "  nessuno")

## 3. Mart

Verifica unicità sulle chiavi del GROUP BY, anni presenti, null e consistenza dei totali con il clean.

In [ ]:
mart_path = MART_DIR / f"{MART_TABLE}.parquet"
if not mart_path.exists():
    raise FileNotFoundError(f"Mart non trovato: {mart_path}. Eseguire: toolkit run mart")

mart = pd.read_parquet(mart_path)
print(f"Righe mart  : {len(mart)}")
print(f"Colonne     : {list(mart.columns)}")
mart.head(3)

In [ ]:
# Estrai chiavi GROUP BY dalla mart.sql per il check di unicità.
# Per query con CTE, cerca GROUP BY solo nella SELECT finale (dopo tutti i CTE).
# Se la SELECT finale non ha GROUP BY, il check di unicità va fatto manualmente.
mart_sql_path = SQL_DIR / Path(cfg["mart"]["tables"][0]["sql"]).name
groupby_keys = []
if mart_sql_path.exists():
    sql_text = mart_sql_path.read_text()

    # Individua dove inizia la SELECT finale (depth 0, dopo eventuali CTE)
    sql_body = sql_text
    if re.search(r'\bwith\b', sql_body, re.IGNORECASE):
        depth = 0
        final_select_pos = None
        for tok in re.finditer(r'[()]|\bselect\b', sql_body, re.IGNORECASE):
            ch = tok.group()
            if ch == '(':
                depth += 1
            elif ch == ')':
                depth -= 1
            elif ch.lower() == 'select' and depth == 0:
                final_select_pos = tok.start()
        if final_select_pos is not None:
            sql_body = sql_body[final_select_pos:]

    match = re.search(r'group\s+by\s+([\d\s,]+)', sql_body, re.IGNORECASE)
    if match:
        positions = [int(p.strip()) for p in match.group(1).split(',')]
        groupby_keys = [mart.columns[i - 1] for i in positions if i <= len(mart.columns)]
    else:
        match = re.search(r'group\s+by\s+((?:[\w.]+(?:\s*,\s*)?)+)', sql_body, re.IGNORECASE)
        if match:
            groupby_keys = [k.strip().split('.')[-1] for k in match.group(1).split(',')]
            groupby_keys = [k for k in groupby_keys if k in mart.columns]

if groupby_keys:
    dups = mart.duplicated(subset=groupby_keys).sum()
    print(f"Chiavi GROUP BY (SELECT finale): {groupby_keys}")
    print(f"Duplicati                       : {dups}")
    assert dups == 0, f"FAIL: {dups} righe duplicate sulle chiavi del mart -- verificare mart.sql"
    print("OK: nessun duplicato sulle chiavi.")
else:
    print("GROUP BY non trovato nella SELECT finale -- check duplicati saltato.")

In [ ]:
if "anno" in mart.columns:
    anni_mart = sorted(mart["anno"].unique())
    print(f"Anni nel mart ({len(anni_mart)}): {anni_mart}")

null_mart = mart.isnull().sum()
print("\nNull per colonna mart:")
print(null_mart[null_mart > 0].to_string() if null_mart.any() else "  nessuno")

In [ ]:
if METRICA in mart.columns and METRICA_CLEAN in clean.columns:
    tot_mart  = mart[METRICA].sum()
    tot_clean = clean[METRICA_CLEAN].sum()
    print(f"Totale mart  ({METRICA})      : {tot_mart:,.2f}")
    print(f"Totale clean ({METRICA_CLEAN}): {tot_clean:,.2f}")
    print("Nota: il mart filtra per COMUNE/PRO -- totali non confrontabili.")
    print("Check non applicabile a questo dataset.")
else:
    print("Nota: il mart filtra per COMUNE/PRO -- totali non confrontabili.")
    print("Check non applicabile a questo dataset.")


In [ ]:
PERIMETRO = "comuni, uscite, 2021-2025, SIOPE aperto"

print(f"Slug            : {SLUG}")
print(f"Anno run        : {ANNO_RUN}")
print(f"Tabella mart    : {MART_TABLE}")
print(f"Metrica mart    : {METRICA}")
print(f"Metrica clean   : {METRICA_CLEAN}")
print(f"Perimetro       : {PERIMETRO}")